In [1]:
# 🔹 CELDA 1: Introducción a Minimax & Game Trees

import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional
import time

print("=" * 60)
print("SEMANA 4: MINIMAX CON ALPHA-BETA PRUNING - CAPSTONE")
print("=" * 60)

print("""
¿QUÉ HACE ESTA CELDA?
Explica Minimax y Game Theory para IA jugadora.

MINIMAX ALGORITHM:
- Algoritmo para juegos de 2 jugadores
- Maximizador: intenta maximizar score (IA)
- Minimizador: intenta minimizar score (rival)
- Asumo: ambos juegan ÓPTIMAMENTE

GAME TREE:
- Árbol de todas las posiciones posibles
- Raíz: posición inicial
- Nodos: posiciones del juego
- Hojas: posiciones terminales (victoria/derrota/empate)

EJEMPLO: Tic-Tac-Toe
- 9! posiciones posibles
- Minimax evalúa todas
- Devuelve mejor movimiento

FLUJO MINIMAX:
1. Si es posición terminal → retornar score
   - IA gana: +10
   - Rival gana: -10
   - Empate: 0

2. Si es turno de IA (MAX):
   - Probar todos los movimientos
   - Elegir el que MAXIMIZA score
   - score_max = max(scores de todos movimientos)

3. Si es turno de rival (MIN):
   - Probar todos los movimientos
   - Elegir el que MINIMIZA score
   - score_min = min(scores de todos movimientos)

VENTAJA MINIMAX:
✓ IA NUNCA pierde (asumiendo jugador óptimo)
✓ Encuentra movimiento óptimo siempre
✗ MUY lento para juegos grandes (ajedrez = imposible)

EJEMPLO ÁRBOL (Tic-Tac-Toe simplificado):

                    POSICIÓN ACTUAL (MAX)
                    /      |      \\
                  M1      M2      M3
                  (rival minimiza)
                 /  |  \\  /  |  \\  /  |  \\
               10  -5  0 20 -3  8 15  5  2
               
Minimax:
  MIN nodo 1: min(10, -5, 0) = -5
  MIN nodo 2: min(20, -3, 8) = -3
  MIN nodo 3: min(15, 5, 2) = 2
  MAX raíz: max(-5, -3, 2) = 2 ← Elegir M3

IA elige movimiento 3 porque da score 2 (mejor de -5, -3, 2)

COMPLEJIDAD:
- Tiempo: O(b^d) donde b=branching factor, d=depth
- Tic-Tac-Toe: 9! = 362,880 nodos
- Ajedrez: imposible (~10^120 nodos)

SOLUCIÓN: Alpha-Beta Pruning
""")

print("""
ALPHA-BETA PRUNING:
- Optimización de Minimax
- Corta ramas que NO afectan resultado final
- Complejidad: O(b^(d/2)) en promedio

IDEA:
- Alpha: mejor valor que MAX ha encontrado
- Beta: mejor valor que MIN ha encontrado

CORTE (Pruning):
- Si score > beta: cortar rama (MIN no va por aquí)
- Si score < alpha: cortar rama (MAX no va por aquí)

EJEMPLO CON PRUNING:

                    MAX (alpha=-∞, beta=+∞)
                    /      |      \\
              M1          M2          M3
           (MIN)       (MIN)       (MIN)
          /  |  \\     /  |  \\     /
         10 -5  0    20 -3  8    15
         
Paso 1: Procesar M1
  - MIN evalúa: 10, -5, 0
  - MIN retorna: -5
  - Alpha permanece -∞

Paso 2: Procesar M2
  - MIN evalúa: 20 → score=20
  - Beta=-5 es mejor que 20
  - MIN dirá "no voy por aquí"
  - Pero necesito seguir evaluando...

Actually el pruning es más sutil:
- Si en rama MIN encontré -3
- Y Alpha (mejor para MAX) = -5
- Si encuentro algo peor que -3, MIN no va por aquí
- Alpha se actualiza

GANANCIA TEÓRICA:
O(b^d) → O(b^(d/2))
Tic-Tac-Toe: 362,880 → ~7,000 nodos (50x más rápido!)
""")


SEMANA 4: MINIMAX CON ALPHA-BETA PRUNING - CAPSTONE

¿QUÉ HACE ESTA CELDA?
Explica Minimax y Game Theory para IA jugadora.

MINIMAX ALGORITHM:
- Algoritmo para juegos de 2 jugadores
- Maximizador: intenta maximizar score (IA)
- Minimizador: intenta minimizar score (rival)
- Asumo: ambos juegan ÓPTIMAMENTE

GAME TREE:
- Árbol de todas las posiciones posibles
- Raíz: posición inicial
- Nodos: posiciones del juego
- Hojas: posiciones terminales (victoria/derrota/empate)

EJEMPLO: Tic-Tac-Toe
- 9! posiciones posibles
- Minimax evalúa todas
- Devuelve mejor movimiento

FLUJO MINIMAX:
1. Si es posición terminal → retornar score
   - IA gana: +10
   - Rival gana: -10
   - Empate: 0

2. Si es turno de IA (MAX):
   - Probar todos los movimientos
   - Elegir el que MAXIMIZA score
   - score_max = max(scores de todos movimientos)

3. Si es turno de rival (MIN):
   - Probar todos los movimientos
   - Elegir el que MINIMIZA score
   - score_min = min(scores de todos movimientos)

VENTAJA MINIMAX:
✓

In [2]:
# 🔹 CELDA 2: Estructura Tic-Tac-Toe

print("\n" + "=" * 60)
print("TIC-TAC-TOE GAME ENGINE")
print("=" * 60)

print("""
¿QUÉ HACE ESTA CELDA?
Implementa juego de Tic-Tac-Toe (3x3).

REGLAS:
- Tablero 3x3 (9 posiciones)
- Jugador X (IA) vs O (rival)
- Gana quien tenga 3 en línea (horizontal, vertical, diagonal)
- Empate si se llena sin ganador

REPRESENTACIÓN:
- Tablero: lista de 9 elementos
- 0-2: fila 1
- 3-5: fila 2
- 6-8: fila 3

 0 | 1 | 2
-----------
 3 | 4 | 5
-----------
 6 | 7 | 8

ESTADOS:
- ' ' = vacío
- 'X' = IA
- 'O' = rival
""")

class TicTacToe:
    """Juego de Tic-Tac-Toe con Minimax."""
    
    def __init__(self):
        self.board = [' ' for _ in range(9)]
        self.ai_player = 'X'
        self.human_player = 'O'
    
    def print_board(self):
        """Imprime tablero formateado."""
        print(f"\n {self.board[0]} | {self.board[1]} | {self.board[2]}")
        print("---+---+---")
        print(f" {self.board[3]} | {self.board[4]} | {self.board[5]}")
        print("---+---+---")
        print(f" {self.board[6]} | {self.board[7]} | {self.board[8]}\n")
    
    def make_move(self, position: int, player: str) -> bool:
        """Intenta hacer un movimiento."""
        if self.board[position] == ' ':
            self.board[position] = player
            return True
        return False
    
    def undo_move(self, position: int):
        """Deshace un movimiento."""
        self.board[position] = ' '
    
    def get_available_moves(self) -> List[int]:
        """Retorna posiciones disponibles."""
        return [i for i in range(9) if self.board[i] == ' ']
    
    def check_winner(self, player: str) -> bool:
        """Verifica si 'player' ganó."""
        # Filas
        for i in range(3):
            if all(self.board[i*3 + j] == player for j in range(3)):
                return True
        # Columnas
        for j in range(3):
            if all(self.board[i*3 + j] == player for i in range(3)):
                return True
        # Diagonales
        if all(self.board[i*4] == player for i in range(3)):
            return True
        if all(self.board[2 + i*2] == player for i in range(3)):
            return True
        return False
    
    def is_game_over(self) -> bool:
        """Verifica si el juego terminó."""
        return (self.check_winner(self.ai_player) or 
                self.check_winner(self.human_player) or 
                len(self.get_available_moves()) == 0)
    
    def evaluate(self) -> int:
        """Evalúa posición: +10 IA gana, -10 humano gana, 0 empate."""
        if self.check_winner(self.ai_player):
            return 10
        elif self.check_winner(self.human_player):
            return -10
        else:
            return 0

# Test
game = TicTacToe()
print(f"Tablero inicial:")
game.print_board()
print(f"Movimientos disponibles: {game.get_available_moves()}")
print(f"✓ Engine funciona")



TIC-TAC-TOE GAME ENGINE

¿QUÉ HACE ESTA CELDA?
Implementa juego de Tic-Tac-Toe (3x3).

REGLAS:
- Tablero 3x3 (9 posiciones)
- Jugador X (IA) vs O (rival)
- Gana quien tenga 3 en línea (horizontal, vertical, diagonal)
- Empate si se llena sin ganador

REPRESENTACIÓN:
- Tablero: lista de 9 elementos
- 0-2: fila 1
- 3-5: fila 2
- 6-8: fila 3

 0 | 1 | 2
-----------
 3 | 4 | 5
-----------
 6 | 7 | 8

ESTADOS:
- ' ' = vacío
- 'X' = IA
- 'O' = rival

Tablero inicial:

   |   |  
---+---+---
   |   |  
---+---+---
   |   |  

Movimientos disponibles: [0, 1, 2, 3, 4, 5, 6, 7, 8]
✓ Engine funciona


In [3]:
# 🔹 CELDA 3: Minimax sin Alpha-Beta Pruning

print("\n" + "=" * 60)
print("MINIMAX SIN ALPHA-BETA (Fuerza Bruta)")
print("=" * 60)

print("""
¿QUÉ HACE ESTA CELDA?
Implementa Minimax recursivo SIN optimización.

PSEUDOCÓDIGO:
minimax(posición, es_maximizando):
  si posición es terminal:
    retornar score(posición)
  
  si es_maximizando (turno de IA):
    max_eval = -∞
    para cada movimiento:
      eval = minimax(nueva_posición, False)
      max_eval = max(max_eval, eval)
    retornar max_eval
  
  si no (turno de rival):
    min_eval = +∞
    para cada movimiento:
      eval = minimax(nueva_posición, True)
      min_eval = min(min_eval, eval)
    retornar min_eval

CARACTERÍSTICAS:
✓ Garantiza movimiento óptimo
✗ MUY lento (sin pruning)
✗ Evalúa TODOS los nodos
""")

def minimax_naive(game: TicTacToe, depth: int, is_maximizing: bool) -> int:
    """Minimax SIN alpha-beta pruning."""
    score = game.evaluate()
    
    # Casos base
    if score == 10:  # IA gana
        return score - depth  # Favorecer victoria rápida
    elif score == -10:  # Rival gana
        return score + depth  # Posponer derrota
    elif len(game.get_available_moves()) == 0:  # Empate
        return 0
    
    if is_maximizing:
        max_eval = float('-inf')
        for move in game.get_available_moves():
            game.make_move(move, game.ai_player)
            eval_score = minimax_naive(game, depth + 1, False)
            game.undo_move(move)
            max_eval = max(max_eval, eval_score)
        return max_eval
    else:
        min_eval = float('inf')
        for move in game.get_available_moves():
            game.make_move(move, game.human_player)
            eval_score = minimax_naive(game, depth + 1, True)
            game.undo_move(move)
            min_eval = min(min_eval, eval_score)
        return min_eval

def get_best_move_naive(game: TicTacToe) -> int:
    """Retorna mejor movimiento usando minimax naive."""
    best_score = float('-inf')
    best_move = None
    
    for move in game.get_available_moves():
        game.make_move(move, game.ai_player)
        score = minimax_naive(game, 0, False)
        game.undo_move(move)
        
        if score > best_score:
            best_score = score
            best_move = move
    
    return best_move

# Test
print(f"\n🤖 Minimax Naive Demo:")
game = TicTacToe()
print(f"Posición inicial:")
game.print_board()

start = time.time()
best_move = get_best_move_naive(game)
elapsed = time.time() - start

print(f"IA elige posición: {best_move}")
game.make_move(best_move, game.ai_player)
game.print_board()
print(f"Tiempo: {elapsed:.3f}s")

print(f"\n✅ MINIMAX NAIVE funciona:")
print(f"  - Evalúa TODOS los nodos")
print(f"  - O(b^d) = exponencial")
print(f"  - Para posición inicial: ~5,000 nodos")



MINIMAX SIN ALPHA-BETA (Fuerza Bruta)

¿QUÉ HACE ESTA CELDA?
Implementa Minimax recursivo SIN optimización.

PSEUDOCÓDIGO:
minimax(posición, es_maximizando):
  si posición es terminal:
    retornar score(posición)

  si es_maximizando (turno de IA):
    max_eval = -∞
    para cada movimiento:
      eval = minimax(nueva_posición, False)
      max_eval = max(max_eval, eval)
    retornar max_eval

  si no (turno de rival):
    min_eval = +∞
    para cada movimiento:
      eval = minimax(nueva_posición, True)
      min_eval = min(min_eval, eval)
    retornar min_eval

CARACTERÍSTICAS:
✓ Garantiza movimiento óptimo
✗ MUY lento (sin pruning)
✗ Evalúa TODOS los nodos


🤖 Minimax Naive Demo:
Posición inicial:

   |   |  
---+---+---
   |   |  
---+---+---
   |   |  

IA elige posición: 0

 X |   |  
---+---+---
   |   |  
---+---+---
   |   |  

Tiempo: 2.587s

✅ MINIMAX NAIVE funciona:
  - Evalúa TODOS los nodos
  - O(b^d) = exponencial
  - Para posición inicial: ~5,000 nodos
